# KẺ MẠO DANH - TACVU1: Pretrained SOTA Pipeline (ResNet-34 @ 384px + Pairwise Ranking + TTA)

Mô hình ResNet-34 tiền huấn luyện (trong whitelist của BTC) tối ưu ở độ phân giải cao 384px, tinh chỉnh theo cặp ảnh, tự động lưu Checkpoint tốt nhất và áp dụng Test-Time Augmentation (TTA).

import os
import gc
import random
import time
import zipfile
from pathlib import Path

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ============================================================
# CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# ============================================================
KAGGLE_DATA = Path('/kaggle/input/datasets/khoileeptit/ca1olp/Ca1/TACVU1/data')
DATA_ROOT = KAGGLE_DATA if KAGGLE_DATA.exists() else Path('data')
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
SPLIT = 'private_test'        # đổi thành 'private_test' ở giai đoạn kiểm tra bí mật

# Mở khóa private_test nếu có mật khẩu
WORK_DATA = ROOT / 'data'
WORK_DATA.mkdir(parents=True, exist_ok=True)
PRIVATE_DIR = DATA_ROOT / 'private_test'

if not (PRIVATE_DIR / 'pairs.csv').exists():
    for pz in [DATA_ROOT / 'private_test.zip', DATA_ROOT.parent / 'private_test.zip']:
        if pz.exists():
            for pwd in [b'225554', b'629436']:
                try:
                    with zipfile.ZipFile(pz, 'r') as zf:
                        zf.extractall(WORK_DATA, pwd=pwd)
                    PRIVATE_DIR = WORK_DATA / 'private_test'
                    print(f'[Data] Giải nén private_test.zip thành công với pass={pwd.decode()}!')
                    break
                except Exception:
                    continue

# Siêu tham số tối ưu chuẩn thi đấu
EPOCHS = 20              # 20 Epochs cho ResNet-34 tiền huấn luyện hội tụ đỉnh cao
IMAGE_SIZE = 384         # 384x384 giữ trọn vẹn từng chi tiết/artifacts của ảnh AI
BATCH_SIZE = 64          # Batch size chuẩn cho GPU H100 / T4
LEARNING_RATE = 2e-4     # LR tối ưu kết hợp Cosine Annealing
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cấu hình] SPLIT: {SPLIT} | DATA_ROOT: {DATA_ROOT}')
print(f'[Cấu hình] Thiết bị: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'[Cấu hình] Độ phân giải: {IMAGE_SIZE}x{IMAGE_SIZE} | Batch size: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LEARNING_RATE}')


In [ ]:
import os
import gc
import random
import time
import zipfile
from pathlib import Path

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

DATA_ROOT = Path('/home/user/TACVU1/data')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data')

SPLIT = 'private_test'

# Mở khóa private_test nếu có mật khẩu
if not (DATA_ROOT / 'private_test' / 'images').exists():
    zip_path = DATA_ROOT / 'private_test.zip'
    if zip_path.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_ROOT, pwd=b'629436')

# Siêu tham số tối ưu chuẩn thi đấu
EPOCHS = 25              # 25 Epochs cho ResNet-34 tiền huấn luyện hội tụ đỉnh cao
IMAGE_SIZE = 384         # 384x384 giữ trọn vẹn từng chi tiết/artifacts của ảnh AI
BATCH_SIZE = 64          # Batch size chuẩn cho GPU H100 / T4
LEARNING_RATE = 2e-4     # LR tối ưu kết hợp Cosine Annealing
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[cấu hình] SPLIT: {SPLIT} | DATA_ROOT: {DATA_ROOT}')
print(f'[cấu hình] Thiết bị: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'[cấu hình] Độ phân giải: {IMAGE_SIZE}x{IMAGE_SIZE} | Batch size: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LEARNING_RATE}')


## 2. Mô hình Pretrained ResNet-34 & Phép biến đổi ảnh

In [ ]:
train_pairs = pd.read_csv(DATA_ROOT / 'train' / 'pairs.csv', dtype={'pair_id': str})
split_path = PRIVATE_DIR if SPLIT == 'private_test' and (PRIVATE_DIR / 'pairs.csv').exists() else (DATA_ROOT / SPLIT)
query = pd.read_csv(split_path / 'pairs.csv', dtype={'pair_id': str})

# Chia 85% train pairs / 15% val pairs cân bằng nhãn
train_df, val_df = train_test_split(
    train_pairs, test_size=0.15, random_state=SEED, stratify=train_pairs['fake_position']
)

train_set = SingleImages(train_df, train_tf)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)

val_pair_set = ImagePairs(val_df, split_dir='train', labeled=True)
val_loader = DataLoader(val_pair_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)

print(f'[dữ liệu] Train: {len(train_df):,} cặp -> {len(train_set):,} ảnh đơn')
print(f'[dữ liệu] Validation: {len(val_df):,} cặp để chấm Pair-Accuracy')
print(f'[dữ liệu] {SPLIT}: {len(query):,} cặp cần dự đoán')


model = ResNet34PairClassifier(pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_pair_acc = 0.0
best_epoch = 0
BEST_MODEL_PATH = ROOT / 'best_model.pth'

print('================== BẮT ĐẦU HUẤN LUYỆN RESNET-34 @ 384px ==================')

for epoch in range(1, EPOCHS + 1):
    # 1. Huấn luyện
    model.train()
    train_loss, train_correct, train_seen = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)

    for images, labels in pbar:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_seen += labels.size(0)

        pbar.set_postfix({
            'loss': f'{train_loss / train_seen:.4f}',
            'acc': f'{train_correct / train_seen:.4f}'
        })

    # 2. Đánh giá trực tiếp trên Validation Pairs (So sánh logit)
    model.eval()
    val_correct, val_seen = 0, 0
    with torch.inference_mode():
        for first, second, targets, _ in val_loader:
            first, second = first.to(DEVICE, non_blocking=True), second.to(DEVICE, non_blocking=True)
            fake_prob_0 = torch.softmax(model(first), 1)[:, 1]
            fake_prob_1 = torch.softmax(model(second), 1)[:, 1]
            pred_positions = (fake_prob_1 > fake_prob_0).cpu().long()
            val_correct += (pred_positions == targets).sum().item()
            val_seen += targets.size(0)

    val_pair_acc = val_correct / val_seen
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    is_best = False
    if val_pair_acc > best_val_pair_acc:
        best_val_pair_acc = val_pair_acc
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        is_best = True

    status_tag = f'⭐ [BEST SAVED -> Pair Acc: {best_val_pair_acc:.4f}]' if is_best else ''
    print(f'Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss/train_seen:.4f} Acc: {train_correct/train_seen:.4f} | '
          f'Val Pair Acc: {val_pair_acc:.4f} | LR: {lr:.6f} {status_tag}')

print(f'\n[huấn luyện] Hoàn tất! Best Model tại Epoch {best_epoch} với Val Pair Accuracy = {best_val_pair_acc:.4f}')


In [ ]:
started = time.time()

# Nạp lại trọng số tốt nhất đã lưu
model = ResNet34PairClassifier(pretrained=False)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

predictions = np.zeros(len(query), dtype=np.int64)
test_split_name = 'private_test' if SPLIT == 'private_test' and (PRIVATE_DIR / 'pairs.csv').exists() else SPLIT
test_dataset = ImagePairs(query, split_dir=test_split_name, labeled=False)
if test_split_name == 'private_test' and (WORK_DATA / 'private_test' / 'pairs.csv').exists():
    test_dataset.split_dir = 'data/private_test'

pair_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'🚀 [TTA 4-Way] Bắt đầu suy luận với 4 phép biến đổi kết hợp (Original, Horizontal Flip, Multi-scale)...')

with torch.inference_mode():
    for first, second, indices in tqdm(pair_loader, desc=f'Dự đoán TTA 4-Way {SPLIT}'):
        first, second = first.to(DEVICE, non_blocking=True), second.to(DEVICE, non_blocking=True)
        
        # 4-Way TTA cho ảnh 0 và ảnh 1
        # 1. Ảnh gốc
        # 2. Lật ngang
        first_flip = torch.flip(first, dims=[-1])
        second_flip = torch.flip(second, dims=[-1])
        
        # 3. Zoom nhẹ 1.05x
        h, w = first.shape[-2:]
        first_scaled = F.interpolate(first, scale_factor=1.05, mode='bilinear', align_corners=False)
        second_scaled = F.interpolate(second, scale_factor=1.05, mode='bilinear', align_corners=False)
        dh, dw = (first_scaled.shape[-2] - h) // 2, (first_scaled.shape[-1] - w) // 2
        first_scaled = first_scaled[:, :, dh:dh+h, dw:dw+w]
        second_scaled = second_scaled[:, :, dh:dh+h, dw:dw+w]
        
        # 4. Zoom + Lật ngang
        first_scaled_flip = torch.flip(first_scaled, dims=[-1])
        second_scaled_flip = torch.flip(second_scaled, dims=[-1])
        
        # Tính trung bình xác suất Fake
        fake_prob_0 = (
            torch.softmax(model(first), 1)[:, 1] +
            torch.softmax(model(first_flip), 1)[:, 1] +
            torch.softmax(model(first_scaled), 1)[:, 1] +
            torch.softmax(model(first_scaled_flip), 1)[:, 1]
        ) / 4.0
        
        fake_prob_1 = (
            torch.softmax(model(second), 1)[:, 1] +
            torch.softmax(model(second_flip), 1)[:, 1] +
            torch.softmax(model(second_scaled), 1)[:, 1] +
            torch.softmax(model(second_scaled_flip), 1)[:, 1]
        ) / 4.0
        
        predictions[indices.numpy()] = (fake_prob_1 > fake_prob_0).cpu().numpy()

sub_file = ROOT / 'submission.csv'
sub_zip = ROOT / f'submission_{SPLIT}.zip'
sub_zip_root = ROOT / 'submission.zip'

submission = pd.DataFrame({'pair_id': query.pair_id, 'fake_position': predictions})
submission.to_csv(sub_file, index=False)

with zipfile.ZipFile(sub_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(sub_file, 'submission.csv')

with zipfile.ZipFile(sub_zip_root, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(sub_file, 'submission.csv')

print(f'[dự đoán] {len(submission):,} cặp trong {time.time() - started:.1f}s')
print(f'[dự đoán] Phân bố nhãn dự đoán: {submission.fake_position.value_counts().to_dict()}')
print(f'🎉 [Nộp bài] Đã tạo thành công {sub_zip} và {sub_zip_root}!')


## 4. Nạp dữ liệu với Stratified Validation Split (85% Train / 15% Val)

In [ ]:
train_pairs = pd.read_csv(DATA_ROOT / 'train' / 'pairs.csv', dtype={'pair_id': str})
query = pd.read_csv(DATA_ROOT / SPLIT / 'pairs.csv', dtype={'pair_id': str})

# Chia 85% train pairs / 15% val pairs cân bằng nhãn
train_df, val_df = train_test_split(
    train_pairs, test_size=0.15, random_state=SEED, stratify=train_pairs['fake_position']
)

train_set = SingleImages(train_df, train_tf)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

val_pair_set = ImagePairs(val_df, split_dir='train', labeled=True)
val_loader = DataLoader(val_pair_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'[dữ liệu] Train: {len(train_df):,} cặp -> {len(train_set):,} ảnh đơn')
print(f'[dữ liệu] Validation: {len(val_df):,} cặp để chấm Pair-Accuracy')
print(f'[dữ liệu] {SPLIT}: {len(query):,} cặp cần dự đoán')


## 5. Huấn luyện & Tự động lưu Checkpoint tốt nhất (Best Pair Accuracy)

In [ ]:
model = ResNet34PairClassifier(pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_pair_acc = 0.0
best_epoch = 0

print('================== BẮT ĐẦU HUẤN LUYỆN RESNET-34 @ 384px ==================')

for epoch in range(1, EPOCHS + 1):
    # 1. Huấn luyện
    model.train()
    train_loss, train_correct, train_seen = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)

    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_seen += labels.size(0)

        pbar.set_postfix({
            'loss': f'{train_loss / train_seen:.4f}',
            'acc': f'{train_correct / train_seen:.4f}'
        })

    # 2. Đánh giá trực tiếp trên Validation Pairs (So sánh logit)
    model.eval()
    val_correct, val_seen = 0, 0
    with torch.inference_mode():
        for first, second, targets, _ in val_loader:
            first, second = first.to(DEVICE), second.to(DEVICE)
            fake_prob_0 = torch.softmax(model(first), 1)[:, 1]
            fake_prob_1 = torch.softmax(model(second), 1)[:, 1]
            pred_positions = (fake_prob_1 > fake_prob_0).cpu().long()
            val_correct += (pred_positions == targets).sum().item()
            val_seen += targets.size(0)

    val_pair_acc = val_correct / val_seen
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    is_best = False
    if val_pair_acc > best_val_pair_acc:
        best_val_pair_acc = val_pair_acc
        best_epoch = epoch
        torch.save(model.state_dict(), 'best_model.pth')
        is_best = True

    status_tag = f'⭐ [BEST SAVED -> Pair Acc: {best_val_pair_acc:.4f}]' if is_best else ''
    print(f'Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss/train_seen:.4f} Acc: {train_correct/train_seen:.4f} | '
          f'Val Pair Acc: {val_pair_acc:.4f} | LR: {lr:.6f} {status_tag}')

print(f'\n[huấn luyện] Hoàn tất! Best Model tại Epoch {best_epoch} với Val Pair Accuracy = {best_val_pair_acc:.4f}')


## 6. Dự đoán với Checkpoint tốt nhất & Test-Time Augmentation (TTA)

In [ ]:
started = time.time()

# Nạp lại trọng số tốt nhất đã lưu
model = ResNet34PairClassifier(pretrained=False)
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

predictions = np.zeros(len(query), dtype=np.int64)
pair_loader = DataLoader(ImagePairs(query, split_dir=SPLIT, labeled=False), batch_size=BATCH_SIZE, shuffle=False)

with torch.inference_mode():
    for first, second, indices in tqdm(pair_loader, desc=f'Dự đoán TTA {SPLIT}'):
        first, second = first.to(DEVICE), second.to(DEVICE)
        
        # TTA: dự đoán ảnh gốc + ảnh lật ngang
        first_flip = torch.flip(first, dims=[-1])
        second_flip = torch.flip(second, dims=[-1])
        
        fake_prob_0 = (torch.softmax(model(first), 1)[:, 1] + torch.softmax(model(first_flip), 1)[:, 1]) / 2.0
        fake_prob_1 = (torch.softmax(model(second), 1)[:, 1] + torch.softmax(model(second_flip), 1)[:, 1]) / 2.0
        
        predictions[indices.numpy()] = (fake_prob_1 > fake_prob_0).cpu().numpy()

submission = pd.DataFrame({'pair_id': query.pair_id, 'fake_position': predictions})
submission.to_csv('submission.csv', index=False)

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.csv')

print(f'[dự đoán] {len(submission):,} cặp trong {time.time() - started:.1f}s')
print(f'[dự đoán] Phân bố nhãn dự đoán: {submission.fake_position.value_counts().to_dict()}')
print('[nộp bài] Đã tạo submission.zip (chứa đúng submission.csv ở thư mục gốc)')
